In [1]:
import pandas as pd
import json
import requests, io, time
import ipywidgets as widgets
from IPython.display import Image, display, clear_output
from PIL import Image as PILImage
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from IPython.display import Audio
import base64

In [2]:
movies = pd.read_csv("data/movies_with_posters.csv")
# quick check rows and columns
print(movies.shape)

movies[["Title", "Year", "Genre", "IMDb_10", "Poster"]].head()

(449, 27)


,Title,Year,Genre,IMDb_10,Poster
0,The Godfather,1972,"Drama, Crime",9.2,https://m.media-amazon.com/images/M/MV5BNGEwYj...
1,Seven Samurai,1954,"Action, Drama",8.6,https://m.media-amazon.com/images/M/MV5BZjliMW...
2,12 Angry Men,1957,Drama,9.0,https://m.media-amazon.com/images/M/MV5BYjE4Nz...
3,Schindler's List,1993,"Drama, History, War",9.0,https://m.media-amazon.com/images/M/MV5BNjM1ZD...
4,The Godfather: Part II,1974,"Drama, Crime",9.0,https://m.media-amazon.com/images/M/MV5BMDIxMz...


In [3]:
# check movie by title
movie = movies[movies["Title"] == "Parasite"].iloc[0]
print(movie["Title"], movie["Year"])
display(Image(url=movie["Poster"], width=150))

Parasite 2019


In [4]:
def show_poster(name):
    match = movies[movies["Title"].str.contains(name, case=False)]
    if match.empty:
        print("movie not found")
        return
    movie = match.iloc[0]
    print(movie["Title"], movie["Year"])
    display(Image(url=movie["Poster"], width=150))

show_poster("2001")

2001: A Space Odyssey 1968


In [5]:
# looks at a movie poster and return main color hex code 
def dominant_color(url):
    img_bytes = requests.get(url, timeout=10).content
    img = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")
    img = img.resize((50, 75)) #shirnk the poster size
    small = img.quantize(colors=10) # reduce it to 10 colors
    count, idx = max(small.getcolors()) # find teh colr that covers the most pixels
    r, g, b = small.getpalette()[idx*3 : idx*3+3]
    return f"#{r:02x}{g:02x}{b:02x}"


In [91]:
print("Movie Title:", movies.loc[100, "Title"])
print("Domiant Color:", dominant_color(movies.loc[100, "Poster"]))

Movie Title: Platoon
Domiant Color: #010101


In [81]:
print(movies.columns.tolist()) 

['Rank', 'Title', 'Year', 'Genre', 'Director', 'Cast', 'Language', 'Plot', 'Awards', 'Production', 'Flickmetrix_Score', 'IMDb_10', 'IMDb_100', 'IMDb_Votes', 'Metacritic', 'Critic_Rating_RT', 'Critic_Reviews', 'Audience_Rating', 'Audience_Reviews', 'Letterboxd', 'Letterboxd_Votes', 'Google_Score', 'Streaming_On', 'RT_URL', 'imdbID', 'Custom_Score', 'Poster', 'Color', 'Family']


In [82]:
# find dominant color for the first 5 movies
for i in range(5):
    c = dominant_color(movies.loc[i, "Poster"])
    display(widgets.HTML(f'{movies.loc[i, "Title"]} <span style="background:{c};padding:2px 30px"></span> {c}'))

HTML(value='The Godfather <span style="background:#000000;padding:2px 30px"></span> #000000')

HTML(value='Seven Samurai <span style="background:#4e4634;padding:2px 30px"></span> #4e4634')

HTML(value='12 Angry Men <span style="background:#ffe404;padding:2px 30px"></span> #ffe404')

HTML(value='Schindler\'s List <span style="background:#010103;padding:2px 30px"></span> #010103')

HTML(value='The Godfather: Part II <span style="background:#000000;padding:2px 30px"></span> #000000')

In [15]:
# turn hex code into color family
import colorsys

def hex_to_rgb(h):
    return tuple(int(h[i:i+2], 16) for i in (1, 3, 5))

def color_family(hex_code):
    r, g, b = hex_to_rgb(hex_code)
    h, s, v = colorsys.rgb_to_hsv(r/255, g/255, b/255)
    h = h * 360

    if v < 0.2:
        return "black"
    if s < 0.15:
        return "white" if v > 0.6 else "black"
    if h < 15 or h >= 345:
        return "red"
    if h < 45:
        return "brown" if v < 0.6 else "orange"
    if h < 70:
        return "yellow"
    if h < 170:
        return "green"
    if h < 255:
        return "blue"
    if h < 290:
        return "purple"
    return "pink"

In [83]:
# labels every movie with its color name and show the counts
movies["Family"] = movies["Color"].apply(color_family)
print(movies["Family"].value_counts())

movies.to_csv("data/movies_with_colors.csv", index=False)

Family
black     159
white      98
blue       65
red        39
brown      27
orange     27
yellow     17
green       9
pink        5
purple      3
Name: count, dtype: int64


In [92]:
# define 10 colors player can pick 
SWATCHES = {
    "red": "#d62828", "orange": "#f77f00", "yellow": "#fcbf49",
    "green": "#2a9d8f", "blue": "#1d4e89", "purple": "#6a4c93",
    "pink": "#e56b9f", "brown": "#7f5539",
    "white": "#f1f1f1", "black": "#111111",
}

# game setup (buttons + memory)
higher_btn = widgets.Button(description="⬆ Higher", button_style="success")
lower_btn = widgets.Button(description="⬇ Lower", button_style="danger")
out = widgets.Output()

movie_a = None
movie_b = None
movie_c = None
movie_d = None

In [93]:
# build movie card: poster, title, genre, IMDb rating and plot
def card(movie, show_rating=True, show_year=True, show_plot=False):
    if show_rating:
        rating = movie["IMDb_10"]
    else:
        rating = "?"

    if show_year:
        year = movie["Year"]
    else:
        year = "????"

    if show_plot:
        plot = f"<p style='width:160px;font-size:12px'>{movie['Plot']}</p>"
    else:
        plot = ""

    html = f"""
        <img src="{movie['Poster']}" width="160"><br>
        <b>{movie['Title']}</b> ({year})<br>
        <i>{movie['Genre']}</i><br>
        IMDb: <b>{rating}</b>
        {plot}
    """
    return widgets.HTML(html)

In [94]:
# Load credentials from your service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

In [95]:
# Fetch an access token (this is what gcloud auth print-access-token returns)
creds.refresh(Request())
token = creds.token

In [96]:
with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

In [97]:
# create soundtrack 
def get_audio(movie):
    # builds a prompt from the movie’s color family, main genre, and title
    prompt = f"{movie['Family']} mood {movie['Genre'].split(', ')[0]} movie soundtrack inspired by {movie['Title']}"
    print("🎧 now playing:", prompt)
    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"{movie['Family']} mood {movie['Genre'].split(', ')[0]} movie soundtrack inspired by {movie['Title']}, the story is about {movie['Plot']}"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    # print(response)
    # print(response.text)
    # print(response.status_code)
    # print(response.headers)
    data = response.json()
    bytedata = data["outputs"][2]["data"]
    return base64.b64decode(bytedata)

In [98]:
# test soundtrack prompt
test = movies.iloc[54]
print(f"{test['Family']} mood {test['Genre'].split(', ')[0]} movie soundtrack inspired by {test['Title']}, the story is about: {test['Plot']}")

blue mood Animation movie soundtrack inspired by Toy Story, the story is about: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


In [99]:
# play soundtrack
def play_announcement(text):
    try:
        mp3 = get_audio(text)
        display(Audio(mp3, autoplay=True))
    except Exception as e:
        print(f"(Couldn't play audio: {e})")

In [101]:
# round 1: pick 2 movies by color 
def pick(family):
    # keeps only movies that matches the clicked swatch
    global movie_a, movie_b
    pool = movies[movies["Family"] == family]

    # game output 
    with out:
        clear_output()
        if len(pool) < 2:
            print(f"not enough {family} movies, try another color")
            return
    # pick 2 random movies and save in movie a and b
    two = pool.sample(2)
    movie_a = two.iloc[0]
    movie_b = two.iloc[1]

    # show title and movie cards sides by side 
    with out:
        print("🎬 ROUND 1: higher or lower?")
        display(widgets.HBox([card(movie_a), card(movie_b, show_rating=False)]))
        print(f"is {movie_b['Title']} rated higher or lower than {movie_a['Title']}?")
        # create higher and lower buttons
        higher = widgets.Button(description="⬆ Higher", button_style="success")
        lower = widgets.Button(description="⬇ Lower", button_style="danger")
        higher.on_click(lambda b: check_rating("higher"))
        lower.on_click(lambda b: check_rating("lower"))
        display(widgets.HBox([higher, lower]))

# check ratings
def check_rating(choice):
    a = movie_a["IMDb_10"]
    b = movie_b["IMDb_10"]

    if b > a:
        answer = "higher"
    elif b < a:
        answer = "lower"
    else:
        answer = "tie"

    with out:
        clear_output()
        display(widgets.HBox([card(movie_a), card(movie_b)]))
        if answer == "tie":
            print("🤝 it's a tie!")
        elif choice == answer:
            print("✅ correct!")
        else:
            print("❌ wrong!")

    if a >= b:
        round2(movie_a)
    else:
        round2(movie_b)

In [102]:
# round 2: older or newer
def round2(winner):
    global movie_c, movie_d
    main_genre = winner["Genre"].split(", ")[0]

    similar = movies[movies["Genre"].str.contains(main_genre)]
    similar = similar[similar["Title"] != movie_a["Title"]]
    similar = similar[similar["Title"] != movie_b["Title"]]

    picks = similar.sample(2)
    movie_c = picks.iloc[0]
    movie_d = picks.iloc[1]

    with out:
        print(f"\n🍿 since {winner['Title']} won, here are 2 {main_genre} picks")
        print("🎬 ROUND 2: which one is older?")
        display(widgets.HBox([card(movie_c, show_year=False), card(movie_d, show_year=False)]))

        btn_c = widgets.Button(description=movie_c["Title"][:20])
        btn_d = widgets.Button(description=movie_d["Title"][:20])
        btn_c.on_click(lambda b: check_age(movie_c))
        btn_d.on_click(lambda b: check_age(movie_d))
        display(widgets.HBox([btn_c, btn_d]))


def check_age(chosen):
    if movie_c["Year"] <= movie_d["Year"]:
        older = movie_c
    else:
        older = movie_d

    with out:
        clear_output()
        display(widgets.HBox([card(movie_c), card(movie_d)]))
        if movie_c["Year"] == movie_d["Year"]:
            print("🤝 same year!")
        elif chosen["Title"] == older["Title"]:
            print("✅ correct!")
        else:
            print("❌ wrong!")

    final_round(older)

In [130]:
# final pick and add audio play 
def final_round(base):
    r = base["Rank"]
    above = movies[movies["Rank"] < r].sort_values("Rank").tail(1)
    below = movies[movies["Rank"] > r].sort_values("Rank").head(1)
    finals = pd.concat([above, below])

    with out:
        print(f"\n🏁 FINAL: the movies ranked right above and below {base['Title']} (#{r})")
        print("pick the one you want to watch tonight:")

        choices = []
        for _, m in finals.iterrows():
            btn = widgets.Button(description="🎬 Watch this", button_style="info")
            btn.on_click(lambda b, m=m: final_pick(m))
            choices.append(widgets.VBox([card(m, show_plot=True), btn]))
        display(widgets.HBox(choices))


def final_pick(movie):
    family = movie["Family"]
    bg = SWATCHES[family]

    if family in ["white", "yellow"]:
        text = "#222"
    else:
        text = "white"

    #cast = ", ".join(str(movie["Cast"]).split(", ")[:3])

    with out:
        display(widgets.HTML(f"""
            <div style="background:{bg}; color:{text}; padding:20px; border-radius:10px;
            display:flex; gap:20px; width:650px; height:auto; margin-top:20px;
            box-sizing:border-box; align-items:flex-start">
                <img src="{movie['Poster']}" style="width:200px; height:auto; border-radius:6px; flex-shrink:0">
                <div>
                    <h2 style="margin-top:0">🍿 Tonight's Movie Pick</h2>
                    <h3>{movie['Title']} ({movie['Year']})</h3>
                    <p><b>Genre:</b> {movie['Genre']}</p>
                    <p><b>Director:</b> {movie['Director']}</p>
                    <p><b>Cast:</b> {movie['Cast']}</p>
                    <p><b>IMDb:</b> {movie['IMDb_10']}</p>
                    <p><b>Rank:</b> #{movie['Rank']}</p>
                    <p>{movie['Plot']}</p>
                </div>
            </div>
        """))
        #print("enjoy your movie night!")
        play_announcement(movie)

In [141]:
# connect buttons + start the game

# reset everything before starting
out.clear_output()
movie_a = None
movie_b = None
movie_c = None
movie_d = None

swatch_btns = []
for name, hex_code in SWATCHES.items():
    btn = widgets.Button(description=name.title(), layout=widgets.Layout(width="100px", height="50px"))
    btn.style.button_color = hex_code
    btn.on_click(lambda b, n=name: pick(n))
    swatch_btns.append(btn)

swatch_grid = widgets.VBox([widgets.HBox(swatch_btns[:5]), widgets.HBox(swatch_btns[5:])])

display(widgets.VBox([
    widgets.HTML("<h3>🎨 Pick a color to start movie night 🎨</h3>"),
    swatch_grid,
    out
]))